In [2]:
import os
import re
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import pandas as pd

# ============================================================
# 0. PROJECT STRUCTURE
# ============================================================

BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "output")
WEIGHT_DIR = os.path.join(BASE_DIR, "weights")
SRC_DIR = os.path.join(BASE_DIR, "src")
SHP_DIR = os.path.join(BASE_DIR, "shp")

for d in [DATA_DIR, OUT_DIR, WEIGHT_DIR, SRC_DIR, SHP_DIR]:
    os.makedirs(d, exist_ok=True)

print("PROJECT STRUCTURE READY.")
print("BASE_DIR:", BASE_DIR)

# ============================================================
# 1. CLIP RASTERS — LGN22 (only use buurtcode)
# ============================================================


CLIP_LGN_DIR = os.path.join(OUT_DIR, "clipped_rasters")
os.makedirs(CLIP_LGN_DIR, exist_ok=True)

print("\n=== CLIPPING LGN28 TIFFS ===")
print("Clipped outputs:", CLIP_LGN_DIR)

shp_path = os.path.join(DATA_DIR, "main_buurten selection.shp")
lgn_path = os.path.join(DATA_DIR, "Reclass_LGN22.tif")

shapes = gpd.read_file(shp_path)

for idx, row in shapes.iterrows():
    geom = [row["geometry"]]

    buurtcode = re.sub(r'\W+', '_', row.get("buurtcode", f"unknown_{idx}"))

    # --- NEW: filename only uses buurtcode ---
    out_name = f"{buurtcode}.tif"
    out_path = os.path.join(CLIP_LGN_DIR, out_name)

    with rasterio.open(lgn_path) as src:
        out_img, out_transform = mask(src, geom, crop=True)
        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "height": out_img.shape[1],
            "width": out_img.shape[2],
            "transform": out_transform
        })

    with rasterio.open(out_path, "w", **meta) as dest:
        dest.write(out_img)

    print("Saved:", out_path)

print("✔ LGN clipping done.")


# ============================================================
# 2. CLIP RASTERS — GREEN 4 TYPES (only use buurtcode)
# ============================================================

CLIP_GREEN_DIR = os.path.join(OUT_DIR, "clipped_rasters2")
os.makedirs(CLIP_GREEN_DIR, exist_ok=True)

print("\n=== CLIPPING GREEN 4+1 TIFFS ===")
print("Clipped outputs:", CLIP_GREEN_DIR)

green_path = os.path.join(DATA_DIR, "Green_4+1.tif")

for idx, row in shapes.iterrows():
    geom = [row["geometry"]]

    buurtcode = re.sub(r'\W+', '_', row.get("buurtcode", f"unknown_{idx}"))

    # --- NEW: filename only uses buurtcode ---
    out_name = f"{buurtcode}.tif"
    out_path = os.path.join(CLIP_GREEN_DIR, out_name)

    with rasterio.open(green_path) as src:
        out_img, out_transform = mask(src, geom, crop=True)
        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "height": out_img.shape[1],
            "width": out_img.shape[2],
            "transform": out_transform
        })

    with rasterio.open(out_path, "w", **meta) as dest:
        dest.write(out_img)

    print("Saved:", out_path)

print("✔ GREEN clipping done.")



# ============================================================
# STEP 2 — 为 LGN + GREEN 两组生成 Fragstats FBT 文件（自动）
# ============================================================

def generate_fbt(CLIP_DIR, FRAG_DIR, fbt_name):
    """
    CLIP_DIR: 输入 TIFF 文件夹
    FRAG_DIR: Fragstats 输出文件夹
    fbt_name: 生成的 FBT 文件名
    """
    os.makedirs(FRAG_DIR, exist_ok=True)

    fbt_path = os.path.join(FRAG_DIR, fbt_name)
    template = ", x, 999, x, x, 1, x, IDF_GeoTIFF"
    lines = []

    for root, dirs, files in os.walk(CLIP_DIR):
        for file in files:
            if file.lower().endswith(".tif"):
                tif_path = os.path.join(root, file)
                lines.append(tif_path + template)

    with open(fbt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"✔ FBT 文件已生成: {fbt_path}")


# -------------------------------
# 生成 LGN 的 FBT
# -------------------------------
CLIP_LGN_DIR = os.path.join(OUT_DIR, "clipped_rasters")
FRAG_LGN_DIR = os.path.join(OUT_DIR, "4frg_5types")
generate_fbt(CLIP_LGN_DIR, FRAG_LGN_DIR, "import_LGN.fbt")

# -------------------------------
# 生成 GREEN 的 FBT
# -------------------------------
CLIP_GREEN_DIR = os.path.join(OUT_DIR, "clipped_rasters2")
FRAG_GREEN_DIR = os.path.join(OUT_DIR, "4frg_4green")
generate_fbt(CLIP_GREEN_DIR, FRAG_GREEN_DIR, "import_GREEN.fbt")

print("⚠️ 请打开 Fragstats 分别加载 import_LGN.fbt 和 import_GREEN.fbt，并运行分析。")

"""
============================================================
   ⚠️ FRAGSTATS PROCESS (TO RUN MANUALLY, NOT AUTOMATED)
============================================================

1. 将 .fbt文件导入Fragstats

2. 运行模型，计算：
   - Class-level: PLAND, AI, IJI
   - Landscape-level: PD, SHDI

3. 导出结果文件：
   - LGN_class.csv
   - LGN_land.csv
   - Green_class.csv
   - Green_land.csv

4. 将它们放入 /data 文件夹中。

下面代码从 FRAGSTATS 结果继续。
============================================================
"""

PROJECT STRUCTURE READY.
BASE_DIR: d:\1st\code\project

=== CLIPPING LGN28 TIFFS ===
Clipped outputs: d:\1st\code\project\output\clipped_rasters
Saved: d:\1st\code\project\output\clipped_rasters\BU00140000.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140001.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140002.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140003.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140004.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140005.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140007.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140008.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140100.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140101.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140102.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140103.tif
Saved: d:\1st\code\project\output\clipped_rasters\BU00140104.tif
Saved: d:\

KeyboardInterrupt: 

In [3]:
import os
import pandas as pd

# ============================================================
# 工具函数：清理列名
# ============================================================
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    去掉列名前后的空格、内部空格和 BOM
    """
    df.columns = (
        df.columns
          .astype(str)
          .str.replace("\ufeff", "", regex=False)   # 去 BOM
          .str.strip()                              # 去前后空格
          .str.replace(" ", "", regex=False)        # 去内部空格
    )
    return df


# ============================================================
# 工具函数：从 LID / FILE 字段中提取 buurtcode
#   支持两种命名：
#   - BU00140000_Binnenstad_Noord_Groningen.tif
#   - BU00140000.tif
# ============================================================
def extract_buurtcode(path_str: str) -> str:
    """
    从 Fragstats 的 LID 字段中提取 buurtcode
    """
    import os
    s = str(path_str)
    filename = os.path.basename(s)          # e.g. BU00140000_Binnenstad_Noord_Groningen.tif
    first = filename.split("_")[0]          # e.g. BU00140000 或 BU00140000.tif
    code = first.split(".")[0]              # 去掉 .tif
    return code


# ============================================================
# 工具函数：在 df 里找到 ID 列（LID / FILE 等）
# ============================================================
def find_id_column(df: pd.DataFrame) -> str:
    """
    在 Fragstats 输出中自动识别 ID 列名：
    优先找 LID、FILE、FID、LAYER，如果都没有就用第一列。
    """
    candidates = ["LID", "FILE", "File", "FID", "Layer", "LAYER"]
    for c in candidates:
        if c in df.columns:
            return c
    # 如果一个都找不到，就用第 1 列
    return df.columns[0]


# ============================================================
# 工具函数：从目录中选“最新”的 *_class.csv / *_land.csv
# 以文件修改时间为准
# ============================================================
def pick_latest_csv(FRAG_DIR: str, suffix: str):
    """
    suffix: '_class.csv' 或 '_land.csv'
    返回：最新文件的完整路径，如果不存在则返回 None
    """
    files = [
        f for f in os.listdir(FRAG_DIR)
        if f.endswith(suffix)
    ]
    if not files:
        return None

    files_full = [os.path.join(FRAG_DIR, f) for f in files]
    latest = max(files_full, key=os.path.getmtime)  # 选修改时间最新的
    return latest


# ============================================================
# 核心函数：处理某个 FRAG_DIR 下的 Fragstats 结果
# ============================================================
def process_fragstats_results(FRAG_DIR, pivot_cols, output_name):
    """
    FRAG_DIR: Fragstats 输出文件夹 (包含 .class / .land)
    pivot_cols: class 层级的指标，例如 ['PLAND','AI','IJI']
    output_name: 最终合并文件名，如 'LGN_merged_class_land.csv'
    """
    print(f"\n=== PROCESSING FRAGSTATS RESULTS: {FRAG_DIR} ===")

    # -----------------------------
    # Step 1 — 把 .class / .land 转成 *_class.csv / *_land.csv
    # -----------------------------
    for file in os.listdir(FRAG_DIR):
        if file.endswith(".class") or file.endswith(".land"):
            input_path = os.path.join(FRAG_DIR, file)
            base = file.rsplit(".", 1)[0]

            if file.endswith(".class"):
                out_name = f"{base}_class.csv"
            else:
                out_name = f"{base}_land.csv"

            output_path = os.path.join(FRAG_DIR, out_name)

            # 读取原始文本
            df_raw = pd.read_csv(input_path, sep=None, engine="python")
            df_raw.to_csv(output_path, index=False, encoding="utf-8-sig")
            print(f"[转换成功] {file} → {out_name}")

    # -----------------------------
    # Step 2 — 选择“最新”的 class / land CSV
    # -----------------------------
    class_path = pick_latest_csv(FRAG_DIR, "_class.csv")
    land_path  = pick_latest_csv(FRAG_DIR, "_land.csv")

    if class_path is None:
        raise FileNotFoundError(f"❌ 在 {FRAG_DIR} 中未找到 *_class.csv 文件。")

    print(f"Class file: {os.path.basename(class_path)}")
    if land_path:
        print(f"Land file:  {os.path.basename(land_path)}")
    else:
        print("⚠️ 此目录没有 land 文件，将只使用 class 结果。")

    # -----------------------------
    # Step 3 — 读取并清理列名
    # -----------------------------
    df_class = pd.read_csv(class_path)
    df_class = clean_columns(df_class)

    df_land = None
    if land_path is not None:
        df_land = pd.read_csv(land_path)
        df_land = clean_columns(df_land)

    # -----------------------------
    # Step 4 — 提取 buurtcode
    # -----------------------------
    id_col_class = find_id_column(df_class)
    df_class["buurtcode"] = df_class[id_col_class].apply(extract_buurtcode)
    df_class = df_class.drop(columns=[id_col_class])

    if df_land is not None:
        id_col_land = find_id_column(df_land)
        df_land["buurtcode"] = df_land[id_col_land].apply(extract_buurtcode)
        df_land = df_land.drop(columns=[id_col_land])

    # -----------------------------
    # Step 5 — 对 class 结果做 pivot （TYPE × 指标）
    # -----------------------------
    df_pivot = df_class.pivot_table(
        index="buurtcode",
        columns="TYPE",
        values=pivot_cols,
        aggfunc="first"
    )

    # MultiIndex 列 → 扁平列名：PLAND-built, AI-water 等
    df_pivot.columns = [
        f"{metric}-{cls}"
        for (metric, cls) in df_pivot.columns
    ]
    df_pivot.columns = df_pivot.columns.str.replace(" ", "", regex=False)
    df_pivot.reset_index(inplace=True)

    # -----------------------------
    # Step 6 — 合并 land 指标（如果有的话）
    # -----------------------------
    if df_land is not None:
        df_merged = df_land.merge(df_pivot, on="buurtcode", how="left")
    else:
        df_merged = df_pivot

    # -----------------------------
    # Step 7 — 输出结果
    # -----------------------------
    output_path = os.path.join(FRAG_DIR, output_name)
    df_merged.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"🎉 输出完成: {output_path}")
    return output_path


# ============================================================
# 4. 运行 LGN & GREEN 的 Fragstats 结果处理
# ============================================================
OUT_DIR = r"d:\1st\code\project\output"  # 你自己的 OUT_DIR

LGN_output = process_fragstats_results(
    FRAG_DIR   = os.path.join(OUT_DIR, "4frg_5types"),
    pivot_cols = ['PLAND', 'IJI', 'AI'],
    output_name = "LGN_merged_class_land.csv"
)

GREEN_output = process_fragstats_results(
    FRAG_DIR   = os.path.join(OUT_DIR, "4frg_4green"),
    pivot_cols = ['PLAND', 'AI'],   # green 没有 IJI
    output_name = "GREEN_merged_class.csv"
)


# ============================================================
# 5. LGN + GREEN 合并
# ============================================================
df_LGN   = pd.read_csv(LGN_output)
df_GREEN = pd.read_csv(GREEN_output)

df_final = df_LGN.merge(df_GREEN, on="buurtcode", how="left")

final_path = os.path.join(OUT_DIR, "form_merged_green_again_all.csv")
df_final.to_csv(final_path, index=False, encoding="utf-8-sig")

print("🎉 最终 LGN + GREEN 合并完成:", final_path)


# ============================================================
# 6. 对合并后的表做变量清理和重命名（按你之前的规则）
# ============================================================
print("\n=== CLEANING MERGED FORM DATA ===")

df = pd.read_csv(final_path)
df.columns = df.columns.str.strip()

df = df.rename(columns={
    'AI-water':  'AI_water',
    'AI-built':  'AI_built',
    'AI-highproportion_mostlyshort': 'AI_Dense_Short',
    'AI-highproportion_mostlytall':  'AI_Dense_Tall',
    'AI-lowproportion_mostlyshort':  'AI_Sparse_Short',
    'AI-lowproportion_mostlytall':   'AI_Sparse_Tall',

    'IJI-built': 'IJI_built',
    'IJI-water': 'IJI_water',
    'IJI-highproportion_mostlyshort': 'IJI_Dense_Short',
    'IJI-highproportion_mostlytall':  'IJI_Dense_Tall',
    'IJI-lowproportion_mostlyshort':  'IJI_Sparse_Short',
    'IJI-lowproportion_mostlytall':   'IJI_Sparse_Tall',

    'PLAND-water':        'PLAND_water',
    'PLAND-built':        'PLAND_built',
    'PLAND-agriculture':  'PLAND_agriculture',
    'PLAND-highproportion_mostlyshort': 'PLAND_Dense_Short',
    'PLAND-highproportion_mostlytall':  'PLAND_Dense_Tall',
    'PLAND-lowproportion_mostlyshort':  'PLAND_Sparse_Short',
    'PLAND-lowproportion_mostlytall':   'PLAND_Sparse_Tall',
})

drop_cols = [
    '0', 'B', 'C', 'D', 'E', 'F', 'G', 'A!',
    'ste_oad', 'ste_mvs', 'a_opp_ha',
    'PD_y', 'CONTAG_y', 'PRD_y', 'SHDI_y', 'AI_y'
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

pattern_cols = [c for c in df.columns if "-others" in c or "-cls_5" in c]
df.drop(columns=pattern_cols, inplace=True)

clean_final_path = os.path.join(OUT_DIR, "form_merged_green_cleaned.csv")
df.to_csv(clean_final_path, index=False, encoding="utf-8-sig")

print("✅ 清理后的最终文件:", clean_final_path)
print(len(df))



=== PROCESSING FRAGSTATS RESULTS: d:\1st\code\project\output\4frg_5types ===
[转换成功] 1205_all.class → 1205_all_class.csv
[转换成功] 1205_all.land → 1205_all_land.csv
Class file: 1205_all_class.csv
Land file:  1205_all_land.csv
🎉 输出完成: d:\1st\code\project\output\4frg_5types\LGN_merged_class_land.csv

=== PROCESSING FRAGSTATS RESULTS: d:\1st\code\project\output\4frg_4green ===
[转换成功] 1205_all.class → 1205_all_class.csv
Class file: 1205_all_class.csv
⚠️ 此目录没有 land 文件，将只使用 class 结果。
🎉 输出完成: d:\1st\code\project\output\4frg_4green\GREEN_merged_class.csv
🎉 最终 LGN + GREEN 合并完成: d:\1st\code\project\output\form_merged_green_again_all.csv

=== CLEANING MERGED FORM DATA ===
✅ 清理后的最终文件: d:\1st\code\project\output\form_merged_green_cleaned.csv
14317
